In [0]:
%sql
CREATE OR REPLACE TABLE synchrony.analytics.report_category_customer AS
SELECT
    c.Customer_ID,
    t.Category,

    SUM(
        CASE WHEN YEAR(t.Transaction_Date) >= 2024
                  AND (
                      (MONTH(t.Transaction_Date) >= 8 AND YEAR(t.Transaction_Date) + 1 = 2025)
                      OR
                      (MONTH(t.Transaction_Date) <= 7 AND YEAR(t.Transaction_Date) = 2025)
                  )
             THEN t.Net_Amount ELSE 0 END
    ) AS FY2025_Total_Spend,

    SUM(
        CASE WHEN (
                      (MONTH(t.Transaction_Date) >= 8 AND YEAR(t.Transaction_Date) + 1 = 2026)
                      OR
                      (MONTH(t.Transaction_Date) <= 7 AND YEAR(t.Transaction_Date) = 2026)
                  )
             THEN t.Net_Amount ELSE 0 END
    ) AS FY2026_Total_Spend,

    SUM(
        CASE WHEN (
                      (MONTH(t.Transaction_Date) >= 8 AND YEAR(t.Transaction_Date) + 1 = 2025)
                      OR
                      (MONTH(t.Transaction_Date) <= 7 AND YEAR(t.Transaction_Date) = 2025)
                  )
             AND t.Payment_Code = 3
             THEN t.Net_Amount ELSE 0 END
    ) AS FY2025_HSIC_Spend,

    SUM(
        CASE WHEN (
                      (MONTH(t.Transaction_Date) >= 8 AND YEAR(t.Transaction_Date) + 1 = 2026)
                      OR
                      (MONTH(t.Transaction_Date) <= 7 AND YEAR(t.Transaction_Date) = 2026)
                  )
             AND t.Payment_Code = 3
             THEN t.Net_Amount ELSE 0 END
    ) AS FY2026_HSIC_Spend

FROM synchrony.analytics.customer_movement_analysis c
JOIN synchrony.analytics.active_transactions t
    ON c.Customer_ID = t.Customer_ID

WHERE c.SoW_Movement = 'SoW_DECLINED'
  AND c.HSIC_Movement = 'HSIC_SPEND_DOWN'
  AND c.Total_Spend_Movement = 'TOTAL_SPEND_UP'

GROUP BY
    c.Customer_ID,
    t.Category;

In [0]:
%sql
CREATE OR REPLACE TABLE synchrony.analytics.report_category_customer AS
SELECT
    *,
    
    CASE
        WHEN FY2025_Total_Spend > 0
        THEN FY2025_HSIC_Spend / FY2025_Total_Spend
        ELSE NULL
    END AS FY2025_SoW,

    CASE
        WHEN FY2026_Total_Spend > 0
        THEN FY2026_HSIC_Spend / FY2026_Total_Spend
        ELSE NULL
    END AS FY2026_SoW,

    FY2026_Total_Spend - FY2025_Total_Spend AS Total_Spend_Change,

    FY2026_HSIC_Spend - FY2025_HSIC_Spend AS HSIC_Spend_Change

FROM synchrony.analytics.report_category_customer;

In [0]:
%sql
SELECT
    Category,
    COUNT(DISTINCT Customer_ID) AS Customer_Count,

    SUM(FY2025_Total_Spend) AS FY2025_Total_Spend,
    SUM(FY2026_Total_Spend) AS FY2026_Total_Spend,

    SUM(FY2025_HSIC_Spend) AS FY2025_HSIC_Spend,
    SUM(FY2026_HSIC_Spend) AS FY2026_HSIC_Spend,

    SUM(Total_Spend_Change) AS Total_Spend_Change,
    SUM(HSIC_Spend_Change) AS HSIC_Spend_Change

FROM synchrony.analytics.report_category_customer

GROUP BY Category

ORDER BY HSIC_Spend_Change ASC;